## DATA CLEANING- Ethnic diversity

In [7]:
#import libraries
import pandas as pd

# Load dataset
df = pd.read_csv('../data_raw/TS021-2021-3.csv')

In [8]:
#quick preview
df.head()

,Lower tier local authorities Code,Lower tier local authorities,Ethnic group (20 categories) Code,Ethnic group (20 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0
1,E06000001,Hartlepool,1,"Asian, Asian British or Asian Welsh: Bangladeshi",278
2,E06000001,Hartlepool,2,"Asian, Asian British or Asian Welsh: Chinese",217
3,E06000001,Hartlepool,3,"Asian, Asian British or Asian Welsh: Indian",335
4,E06000001,Hartlepool,4,"Asian, Asian British or Asian Welsh: Pakistani",297


In [9]:
#view columns
df.columns

Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Ethnic group (20 categories) Code', 'Ethnic group (20 categories)',
       'Observation'],
      dtype='object')

In [10]:
#confirm data types
df.dtypes

Lower tier local authorities Code    object
Lower tier local authorities         object
Ethnic group (20 categories) Code     int64
Ethnic group (20 categories)         object
Observation                           int64
dtype: object

In [11]:
#confirm completeness
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6620 entries, 0 to 6619
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Lower tier local authorities Code  6620 non-null   object
 1   Lower tier local authorities       6620 non-null   object
 2   Ethnic group (20 categories) Code  6620 non-null   int64 
 3   Ethnic group (20 categories)       6620 non-null   object
 4   Observation                        6620 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 258.7+ KB


In [13]:
#rename columns
df = df.rename(columns={
    'Lower tier local authorities': 'borough',
    'Ethnic group (20 categories)': 'ethnic_group',
    'Observation': 'population'
})

In [14]:
#drop not needed
df = df[df['ethnic_group'] != 'Total: All usual residents']

In [15]:
#filter london
london_boroughs = [
    'Camden', 'Greenwich', 'Hackney', 'Hammersmith and Fulham',
    'Islington', 'Kensington and Chelsea', 'Lambeth', 'Lewisham',
    'Southwark', 'Tower Hamlets', 'Wandsworth', 'Westminster',
    'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Croydon', 'Ealing', 'Enfield', 'Haringey',
    'Harrow', 'Havering', 'Hillingdon', 'Hounslow',
    'Kingston upon Thames', 'Merton', 'Newham', 'Redbridge',
    'Richmond upon Thames', 'Sutton', 'Waltham Forest',
    'City of London'
]

df = df[df['borough'].isin(london_boroughs)]

In [16]:
#change to wide format
df_pivot = df.pivot_table(
    index='borough',
    columns='ethnic_group',
    values='population',
    aggfunc='sum'
).reset_index()

In [18]:
#calculate total population
df_pivot['total'] = df_pivot.iloc[:, 1:].sum(axis=1)

In [20]:
#calculate proprtions
proportions = df_pivot.iloc[:, 1:-1].div(df_pivot['total'], axis=0)

In [21]:
#calculate diversity index
df_pivot['ethnic_diversity'] = 1 - (proportions ** 2).sum(axis=1)

In [22]:
#final output
df_clean = df_pivot[['borough', 'ethnic_diversity']]

df_clean = df_clean.sort_values(by='borough').reset_index(drop=True)

In [25]:
#validate
print("Rows:", len(df_clean))
df_clean.head(33)

Rows: 33


ethnic_group,borough,ethnic_diversity
0,Barking and Dagenham,0.960028
1,Barnet,0.952454
2,Bexley,0.891958
3,Brent,0.971794
4,Bromley,0.886308
5,Camden,0.953160
6,City of London,0.937560
7,Croydon,0.954759
8,Ealing,0.967722
9,Enfield,0.958713


In [24]:
#save
df_clean.to_csv('../data_clean/ethnic_diversity.csv', index=False)